# Notebook local : Pipeline NLP CREMMA Medieval

Version locale du notebook Kaggle : pas de clone Git, pas de Kaggle Secrets,
pas de sync S3 -- on utilise directement le code et les donnees deja presents
dans ce repo (branche `nlp-pipeline`).

Flux : validate -> eda -> review-queue -> normalize-contract -> correct ->
relative-eval -> lexical-check -> split.

## 1. Verifier que le CLI et les donnees sont accessibles

In [12]:
from pathlib import Path
import os

# Remonte a la racine du repo si le kernel demarre dans notebooks/
if Path("nlp_pipeline").is_dir():
    project_root = Path(".").resolve()
elif Path("../nlp_pipeline").is_dir():
    os.chdir("..")
    project_root = Path(".").resolve()
else:
    raise FileNotFoundError("nlp_pipeline/ introuvable depuis le working directory courant")

cli_path = project_root / "nlp_pipeline" / "nlp_cli.py"
dictionary_path = project_root / "data" / "dictionary" / "dictionnaire_ancien_francais.json"
print("Project root:", project_root)
print("CLI exists:", cli_path.exists())
print("Dictionary exists:", dictionary_path.exists())

sample_docs = sorted((project_root / "data" / "nlp_output").rglob("*.json"))
print("Sample HTR contracts found:", len(sample_docs))
sample_doc = sample_docs[0] if sample_docs else None
print("Using sample:", sample_doc)

Project root: C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026
CLI exists: True
Dictionary exists: True
Sample HTR contracts found: 129
Using sample: C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json


## 2. Lancer la pipeline NLP etape par etape

Chaque commande est appelee en sous-processus pour eviter tout souci de `sys.path`.

In [13]:
import subprocess


def run_cli(*args):
    cmd = ["python", str(cli_path), *args]
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
    print("$ ", " ".join(cmd))
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [14]:
# 2.1 Validation du contrat HTR
run_cli("validate", "--input", str(sample_doc))

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py validate --input C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json
OK: C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json
Validation OK: all contracts are valid.



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'validate', '--input', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json'], returncode=0, stdout='OK: C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json\nValidation OK: all contracts are valid.\n', stderr='')

In [15]:
# 2.2 EDA : statistiques de confiance et longueur de ligne
run_cli("eda", "--input", str(sample_doc), "--output", "data/eda_report.json")

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py eda --input C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json --output data/eda_report.json
{
  "n_lines": 157,
  "mean_confidence": 0.9574229299363058,
  "median_line_length": 30,
  "needs_review_rate": 0.03184713375796178,
  "abbr_per_line": 0.3885350318471338,
  "short_line_rate_lt_10": 0.03184713375796178,
  "confidence_quartiles": {
    "q1": 0.9701,
    "q2": 0.9844,
    "q3": 0.9945
  },
  "confidence_bins": {
    "0.0-0.6": 4,
    "0.6-0.7": 0,
    "0.7-0.8": 0,
    "0.8-0.9": 0,
    "0.9-1.0": 153
  }
}



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'eda', '--input', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json', '--output', 'data/eda_report.json'], returncode=0, stdout='{\n  "n_lines": 157,\n  "mean_confidence": 0.9574229299363058,\n  "median_line_length": 30,\n  "needs_review_rate": 0.03184713375796178,\n  "abbr_per_line": 0.3885350318471338,\n  "short_line_rate_lt_10": 0.03184713375796178,\n  "confidence_quartiles": {\n    "q1": 0.9701,\n    "q2": 0.9844,\n    "q3": 0.9945\n  },\n  "confidence_bins": {\n    "0.0-0.6": 4,\n    "0.6-0.7": 0,\n    "0.7-0.8": 0,\n    "0.8-0.9": 0,\n    "0.9-1.0": 153\n  }\n}\n', stderr='')

In [16]:
# 2.3 File de relecture : tri par confiance
run_cli(
    "review-queue",
    "--input", str(sample_doc),
    "--csv-output", "data/review/review_queue.csv",
    "--json-output", "data/review/review_buckets.json",
)

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py review-queue --input C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json --csv-output data/review/review_queue.csv --json-output data/review/review_buckets.json
Direct : 152
Review : 1
Exclude: 4
CSV review queue: data/review/review_queue.csv



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'review-queue', '--input', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json', '--csv-output', 'data/review/review_queue.csv', '--json-output', 'data/review/review_buckets.json'], returncode=0, stdout='Direct : 152\nReview : 1\nExclude: 4\nCSV review queue: data/review/review_queue.csv\n', stderr='')

In [17]:
# 2.4 Normalisation du francais medieval (u/v, i/j, abreviations)
run_cli(
    "normalize-contract",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_normalized.json",
)

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py normalize-contract --input C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json --output data/contracts/contract_normalized.json
Normalized contract saved to: data\contracts\contract_normalized.json



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'normalize-contract', '--input', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json', '--output', 'data/contracts/contract_normalized.json'], returncode=0, stdout='Normalized contract saved to: data\\contracts\\contract_normalized.json\n', stderr='')

In [18]:
# 2.5 Correction guidee par confiance (heuristique par defaut, sans MLM)
run_cli(
    "correct",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_corrected.json",
    "--log-output", "data/review/correction_log.jsonl",
)

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py correct --input C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\nlp_output\btv1b100336071\btv1b100336071_f10.json --output data/contracts/contract_corrected.json --log-output data/review/correction_log.jsonl
Corrections applied: 0
Updated contract   : data\contracts\contract_corrected.json
Correction log     : data/review/correction_log.jsonl



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'correct', '--input', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\nlp_output\\btv1b100336071\\btv1b100336071_f10.json', '--output', 'data/contracts/contract_corrected.json', '--log-output', 'data/review/correction_log.jsonl'], returncode=0, stdout='Corrections applied: 0\nUpdated contract   : data\\contracts\\contract_corrected.json\nCorrection log     : data/review/correction_log.jsonl\n', stderr='')

In [19]:
# 2.5bis Variante avec modele de langue masque CamemBERT (necessite: pip install transformers sentencepiece)
# run_cli(
#     "correct",
#     "--input", str(sample_doc),
#     "--output", "data/contracts/contract_corrected_mlm.json",
#     "--log-output", "data/review/correction_log_mlm.jsonl",
#     "--mlm-model", "almanach/camembert-base",
#     "--mlm-device", "auto",
# )

## 3. Evaluation relative (sans verite terrain)

`data/nlp_output/` ne contient que des predictions HTR brutes, pas de transcription
validee manuellement -- comparer "avant/apres" a une fausse reference (le texte brut
lui-meme) n'a pas de sens (CER avant = 0 par construction).

A la place, `relative-eval` calcule le **CER pairwise moyen** entre plusieurs variantes
d'une meme ligne (`raw`, `text_normalized`, `corrected`).

In [20]:
import csv
import json

raw_contract = json.loads(Path(sample_doc).read_text(encoding="utf-8"))
normalized_contract = json.loads(Path("data/contracts/contract_normalized.json").read_text(encoding="utf-8"))
corrected_contract = json.loads(Path("data/contracts/contract_corrected.json").read_text(encoding="utf-8"))

raw_lines = [line["text"] for page in raw_contract.get("pages", []) for line in page.get("lines", [])]
normalized_lines = [
    line.get("normalized_text", line["text"])
    for page in normalized_contract.get("pages", [])
    for line in page.get("lines", [])
]
corrected_lines = [line["text"] for page in corrected_contract.get("pages", []) for line in page.get("lines", [])]

rows = [
    {"raw": raw, "text_normalized": norm, "corrected": corr}
    for raw, norm, corr in zip(raw_lines, normalized_lines, corrected_lines)
]
relative_csv = Path("data/review/relative_eval_sample.csv")
with relative_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["raw", "text_normalized", "corrected"])
    writer.writeheader()
    writer.writerows(rows)

run_cli("relative-eval", "--csv-input", str(relative_csv))

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py relative-eval --csv-input data\review\relative_eval_sample.csv
Average pairwise CER across variants: 0.0565
Rows evaluated: 155



CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'relative-eval', '--csv-input', 'data\\review\\relative_eval_sample.csv'], returncode=0, stdout='Average pairwise CER across variants: 0.0565\nRows evaluated: 155\n', stderr='')

## 4. Detection lexicale via le dictionnaire d'ancien francais

Contrairement a `detect-normalization` (qui repere seulement des marqueurs
d'abreviation typographiques comme `~`, `⁊`, `ꝑ`), `lexical-check` verifie si
chaque token existe dans `dictionnaire_ancien_francais.json` (Wiktionary +
lexique CLTK). Les tokens absents sont de vraies erreurs lexicales potentielles.

In [21]:
run_cli(
    "lexical-check",
    "--dictionary", str(dictionary_path),
    "--output-dir", "data/nlp_output",
    "--top-n", "30",
    "--output", "data/lexical_report.json",
)

$  python C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\nlp_pipeline\nlp_cli.py lexical-check --dictionary C:\Users\djame\Documents\GitHub\htr-cremma-medieval-2026\data\dictionary\dictionnaire_ancien_francais.json --output-dir data/nlp_output --top-n 30 --output data/lexical_report.json
{
  "total_tokens": 26900,
  "total_counts": 89179,
  "dictionary_size": 54921,
  "unknown_tokens": 25827,
  "top_unknown": [
    {
      "token": "⁊",
      "count": 3103
    },
    {
      "token": "q̃",
      "count": 509
    },
    {
      "token": "e",
      "count": 483
    },
    {
      "token": "est",
      "count": 473
    },
    {
      "token": "ce",
      "count": 447
    },
    {
      "token": "des",
      "count": 343
    },
    {
      "token": "l",
      "count": 307
    },
    {
      "token": "s",
      "count": 307
    },
    {
      "token": "ie",
      "count": 291
    },
    {
      "token": "d",
      "count": 280
    },
    {
      "token": "qͥ",
      "count": 279
 

CompletedProcess(args=['python', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\nlp_pipeline\\nlp_cli.py', 'lexical-check', '--dictionary', 'C:\\Users\\djame\\Documents\\GitHub\\htr-cremma-medieval-2026\\data\\dictionary\\dictionnaire_ancien_francais.json', '--output-dir', 'data/nlp_output', '--top-n', '30', '--output', 'data/lexical_report.json'], returncode=0, stdout='{\n  "total_tokens": 26900,\n  "total_counts": 89179,\n  "dictionary_size": 54921,\n  "unknown_tokens": 25827,\n  "top_unknown": [\n    {\n      "token": "⁊",\n      "count": 3103\n    },\n    {\n      "token": "q̃",\n      "count": 509\n    },\n    {\n      "token": "e",\n      "count": 483\n    },\n    {\n      "token": "est",\n      "count": 473\n    },\n    {\n      "token": "ce",\n      "count": 447\n    },\n    {\n      "token": "des",\n      "count": 343\n    },\n    {\n      "token": "l",\n      "count": 307\n    },\n    {\n      "token": "s",\n      "count": 307\n    },\n    {\n      "token": "

## 5. Push des resultats vers S3 (optionnel)

In [ ]:
sync_script = project_root / "nlp_pipeline" / "sync_to_s3.py"
subprocess.run(["python", str(sync_script)], capture_output=True, text=True)